# Simple CNN vs ResNet-18 on CIFAR-10
### Baseline mini-study: no augmentation, SGD+momentum for both

Two architectures,
one augmentation level ("none"), one optimizer ("SGD with momentum"), one seed.

It is written config-first, so that turning it into the full mid-term design later is
mostly adding entries to a dictionary, not rewriting the pipeline. The last section
walks through exactly what that extension looks like.


In [2]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


Using device: cuda


### Why we seed

Training a neural network involves randomness at several points: the weights start at random values, the data gets shuffled into a different batch order each epoch, and if you add augmentation later, which crop or flip gets applied is also random. Without fixing that randomness, running the exact same code twice gives you two different models with two different accuracies, and you'd have no way to tell whether a change you made actually helped, or you just got a luckier random draw.

A "seed" is the starting value for a random number generator. Same seed in, same sequence of "random" numbers out, every time. The reason there are four calls instead of one is that Python, NumPy, and PyTorch each keep their own independent random number generator, and PyTorch keeps a separate one again for CPU versus GPU. Your training loop touches all of them: PyTorch's own generator for weight initialization, NumPy's for the stratified split you'll see in the next section, Python's `random` in case any library or transform uses it internally, and the CUDA generator specifically for anything running on the GPU. Seeding only some of them would leave a hidden source of randomness that quietly breaks reproducibility.

This is directly graded in your mid-term. Each of your 7 configurations gets trained 3 times with 3 different seeds, and the whole point is that the only thing that changes between those 3 runs is what the seed controls (initialization, shuffling, augmentation draws), while the data split stays fixed. That's what tells you whether a gap between two configurations is a real effect or just noise. `set_seed` is called once at the very start of `train_one_config`, before the model is built, which is exactly why it has to happen there and not once at the top of the notebook: it needs to reset the generators fresh for every single run, so run 2 and run 3 are genuinely independent draws rather than a continuation of run 1's random stream.

**Caveat:** even fully seeded, GPU results can be very slightly non-deterministic
(cuDNN parallelism). That's fine here, seed-level reproducibility is what's required,
not bit-for-bit identical results.

## Data: CIFAR-10 with one fixed train / val / test split

The assignment requires the validation split to be created once, before any experiments,
and reused unchanged for every configuration. We build it here, outside the training loop,
so it really is shared rather than accidentally re-randomized per run.


In [2]:
CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

#This tuple is just a lookup table so you can turn a prediction like 3 back into "cat" when printing results
# or reading a confusion matrix. The order matters.

# No augmentation: only the deterministic resize/normalize transform.
# This is also the eval transform, used for validation and test everywhere,
# and it is never touched by the augmentation study later.
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=eval_transform)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=eval_transform)

# Stratified 90/10 train/val split, created once with a fixed random_state,
# reused for every config below (including future ones).
targets = np.array(full_trainset.targets)
train_idx, val_idx = train_test_split(
    np.arange(len(full_trainset)), test_size=0.10, stratify=targets, random_state=0
)
val_subset = Subset(full_trainset, val_idx)

print(f"train: {len(train_idx)}  val: {len(val_subset)}  test: {len(testset)}")

BATCH_SIZE = 128
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
valloader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


100%|██████████| 170M/170M [42:17<00:00, 67.2kB/s]   


train: 45000  val: 5000  test: 10000


## Architectures

Two levels for this mini study:

- **SimpleCNN**: a small from-scratch network, 3 conv blocks then a small classifier head.
- **ResNet-18**: `torchvision.models.resnet18(weights=None, ...)`, trained from random
  weights, not pretrained. ImageNet-pretrained weights are off-limits for the mid-term
  because they push every architecture toward the same accuracy and hide the differences
  you are meant to measure.

ResNet-18 is designed for 224x224 ImageNet images, so its stem needs adapting for 32x32
CIFAR input: the 7x7 stride-2 first conv becomes a 3x3 stride-1 conv, and the first
max-pool is removed. Without this the image is downsampled to almost nothing before the
first residual block even runs.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32 -> 16
        x = self.pool(F.relu(self.conv2(x)))   # 16 -> 8
        x = self.pool(F.relu(self.conv3(x)))   # 8 -> 4
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)


def build_resnet18(num_classes=10):
    model = torchvision.models.resnet18(weights=None, num_classes=num_classes)
    # Adapt the stem for 32x32 inputs instead of 224x224 ImageNet inputs.
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def build_model(architecture):
    if architecture == "simple_cnn":
        model = SimpleCNN()
    elif architecture == "resnet18":
        model = build_resnet18()
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    return model.to(DEVICE)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [3]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32 -> 16
        x = self.pool(F.relu(self.conv2(x)))   # 16 -> 8
        x = self.pool(F.relu(self.conv3(x)))   # 8 -> 4
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)


def build_resnet18(num_classes=10):
    model = torchvision.models.resnet18(weights=None, num_classes=num_classes)
    # Adapt the stem for 32x32 inputs instead of 224x224 ImageNet inputs.
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def build_densenet121(num_classes=10):
    model = torchvision.models.densenet121(weights=None, num_classes=num_classes)
    # Same adaptation as ResNet-18: smaller first conv, no initial downsampling,
    # so a 32x32 image isn't reduced to a handful of pixels before the first
    # dense block runs. Spatial resolution then goes 32 -> 16 -> 8 -> 4 across
    # the three transition layers, landing at 4x4 before the classifier --
    # the same final resolution as the adapted ResNet-18 above.
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()
    return model


def build_model(architecture):
    if architecture == "simple_cnn":
        model = SimpleCNN()
    elif architecture == "resnet18":
        model = build_resnet18()
    elif architecture == "densenet121":
        model = build_densenet121()
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    return model.to(DEVICE)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Augmentation and optimizer, written as swappable pieces

Only one level of each is implemented right now: `"none"` for augmentation and
`"sgd_momentum"` for the optimizer, matching what you asked for. They are small factory
functions on purpose, so that adding the other two levels of each later is adding one
`if` branch, not restructuring anything.


In [ ]:
def build_train_transform(augmentation):
    if augmentation == "none":
        return eval_transform
    # Extend here for the mid-term augmentation study, e.g.:
    # if augmentation == "crop_flip":
    #     return T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    #                        T.ToTensor(), T.Normalize(...)])
    # if augmentation == "randaugment":
    #     return T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    #                        T.RandAugment(), T.ToTensor(), T.Normalize(...)])
    raise ValueError(f"Unknown augmentation: {augmentation}")


def build_optimizer(optimizer_name, params):
    if optimizer_name == "sgd_momentum":
        return optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=5e-4)
    # Extend here for the mid-term optimizer study, e.g.:
    # if optimizer_name == "adam":
    #     return optim.Adam(params, lr=1e-3, weight_decay=5e-4)
    # if optimizer_name == "adamw":
    #     return optim.AdamW(params, lr=1e-3, weight_decay=5e-4)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")


In [4]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)


def build_train_transform(augmentation):
    if augmentation == "none":
        return eval_transform  # ToTensor + Normalize only -- the control arm
    if augmentation == "crop_flip":
        return T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
        ])
    if augmentation == "strong":
        return T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.RandAugment(),
            T.ToTensor(),
            T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
            T.RandomErasing(p=0.5),
        ])
    raise ValueError(f"Unknown augmentation: {augmentation}")


def build_optimizer(optimizer_name, params):
    if optimizer_name == "sgd_momentum":
        return optim.SGD(params, lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    if optimizer_name == "adam":
        return optim.Adam(params, lr=1e-3, weight_decay=5e-4)
    if optimizer_name == "adamw":
        return optim.AdamW(params, lr=1e-3, weight_decay=5e-4)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")

## Shared train / evaluate loop

One training function used by every configuration: cross-entropy loss, early stopping
and checkpoint selection on validation loss (never on the test set), and a fixed epoch
budget so it can be repeated across configs and seeds without becoming unaffordable.


In [ ]:
EPOCHS = 20
PATIENCE = 5

criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, loss_sum, n = [], [], 0.0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss_sum += criterion(outputs, labels).item() * labels.size(0)
        n += labels.size(0)
        all_preds.append(outputs.argmax(1).cpu())
        all_labels.append(labels.cpu())
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return {
        "loss": loss_sum / n,
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "preds": preds,
        "labels": labels,
    }


def train_one_config(architecture, augmentation, optimizer_name, seed):
    set_seed(seed)
    model = build_model(architecture)
    optimizer = build_optimizer(optimizer_name, model.parameters())

    train_transform = build_train_transform(augmentation)
    train_ds = torchvision.datasets.CIFAR10(root="./data", train=True, download=False, transform=train_transform)
    train_ds = Subset(train_ds, train_idx)   # same stratified indices every time
    trainloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss, best_state, patience_left = float("inf"), None, PATIENCE

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        t0 = time.time()
        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            seen += labels.size(0)

        train_loss = running_loss / seen
        val_metrics = evaluate(model, valloader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])

        print(f"[{architecture} seed{seed}] epoch {epoch:>2}/{EPOCHS}  "
              f"train_loss {train_loss:.4f}  val_loss {val_metrics['loss']:.4f}  "
              f"val_acc {val_metrics['accuracy']:.4f}  ({time.time()-t0:.1f}s)")

        # Early stopping and model selection on validation loss, never on the test set.
        if val_metrics["loss"] < best_val_loss:
            best_val_loss, best_state, patience_left = val_metrics["loss"], model.state_dict(), PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"  early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_metrics = evaluate(model, testloader)
    return {
        "architecture": architecture,
        "augmentation": augmentation,
        "optimizer": optimizer_name,
        "seed": seed,
        "params": count_params(model),
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "history": history,
        "test_preds": test_metrics["preds"],
        "test_labels": test_metrics["labels"],
    }


In [7]:
import copy
import time

EPOCHS = 10
PATIENCE = 5

criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, loss_sum, n = [], [], 0.0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss_sum += criterion(outputs, labels).item() * labels.size(0)
        n += labels.size(0)
        all_preds.append(outputs.argmax(1).cpu())
        all_labels.append(labels.cpu())
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return {
        "loss": loss_sum / n,
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "preds": preds,
        "labels": labels,
    }


def train_one_config(architecture, augmentation, optimizer_name, seed):
    set_seed(seed)
    model = build_model(architecture)
    optimizer = build_optimizer(optimizer_name, model.parameters())
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    train_transform = build_train_transform(augmentation)
    train_ds = torchvision.datasets.CIFAR10(root="./data", train=True, download=False, transform=train_transform)
    train_ds = Subset(train_ds, train_idx)   # same stratified indices every time
    trainloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss, best_state, patience_left = float("inf"), None, PATIENCE
    run_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, running_correct, seen = 0.0, 0, 0
        t0 = time.time()
        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            running_correct += (outputs.argmax(1) == labels).sum().item()
            seen += labels.size(0)
        scheduler.step()

        train_loss = running_loss / seen
        train_acc = running_correct / seen
        val_metrics = evaluate(model, valloader)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])

        print(f"[{architecture} seed{seed}] epoch {epoch:>2}/{EPOCHS}  "
              f"train_loss {train_loss:.4f}  train_acc {train_acc:.4f}  "
              f"val_loss {val_metrics['loss']:.4f}  val_acc {val_metrics['accuracy']:.4f}  "
              f"({time.time()-t0:.1f}s)")

        # Early stopping and model selection on validation loss, never on the test set.
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"  early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_metrics = evaluate(model, testloader)
    return {
        "architecture": architecture,
        "augmentation": augmentation,
        "optimizer": optimizer_name,
        "seed": seed,
        "params": count_params(model),
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "train_time_sec": time.time() - run_start,
        "history": history,
        "test_preds": test_metrics["preds"],
        "test_labels": test_metrics["labels"],
    }

## The two configurations you asked for

This *is* the experiment-design section of the mid-term, just with 2 configs and 1 seed
instead of 7 configs and 3 seeds. Each entry only names an architecture, an augmentation,
and an optimizer; everything else is identical because it is controlled by the shared
functions above, which is exactly the point of a controlled comparison.


In [ ]:
CONFIGS = {
    "simple_cnn": {"architecture": "simple_cnn", "augmentation": "none", "optimizer": "sgd_momentum"},
    "resnet18":   {"architecture": "resnet18",   "augmentation": "none", "optimizer": "sgd_momentum"},
}
SEEDS = [0]   # for the real mid-term: SEEDS = [0, 1, 2]

results = []
for name, cfg in CONFIGS.items():
    for seed in SEEDS:
        result = train_one_config(cfg["architecture"], cfg["augmentation"], cfg["optimizer"], seed)
        result["config"] = name
        results.append(result)


In [8]:
CONFIGS = {
    "C1_resnet18_baseline": {"architecture": "resnet18",     "augmentation": "crop_flip", "optimizer": "sgd_momentum"},
    "C2_simple_cnn":        {"architecture": "simple_cnn",   "augmentation": "crop_flip", "optimizer": "sgd_momentum"},
    "C3_densenet121":       {"architecture": "densenet121",  "augmentation": "crop_flip", "optimizer": "sgd_momentum"},
    "C4_aug_none":          {"architecture": "resnet18",     "augmentation": "none",      "optimizer": "sgd_momentum"},
    "C5_aug_strong":        {"architecture": "resnet18",     "augmentation": "strong",    "optimizer": "sgd_momentum"},
    "C6_opt_adam":          {"architecture": "resnet18",     "augmentation": "crop_flip", "optimizer": "adam"},
    "C7_opt_adamw":         {"architecture": "resnet18",     "augmentation": "crop_flip", "optimizer": "adamw"},
}
SEEDS = [0]   # for the real mid-term: SEEDS = [0, 1, 2]

results = []
total_runs = len(CONFIGS) * len(SEEDS)
run_num = 0

for name, cfg in CONFIGS.items():
    for seed in SEEDS:
        run_num += 1
        print(f"\n=== run {run_num}/{total_runs}: {name}  "
              f"(arch={cfg['architecture']}, aug={cfg['augmentation']}, "
              f"opt={cfg['optimizer']}, seed={seed}) ===", flush=True)
        result = train_one_config(cfg["architecture"], cfg["augmentation"], cfg["optimizer"], seed)
        result["config"] = name
        results.append(result)

print(f"\nDone: {len(results)}/{total_runs} runs completed.")


=== run 1/7: C1_resnet18_baseline  (arch=resnet18, aug=crop_flip, opt=sgd_momentum, seed=0) ===
[resnet18 seed0] epoch  1/10  train_loss 2.1375  train_acc 0.2587  val_loss 1.6444  val_acc 0.3872  (40.9s)
[resnet18 seed0] epoch  2/10  train_loss 1.5462  train_acc 0.4286  val_loss 1.4639  val_acc 0.4828  (37.0s)
[resnet18 seed0] epoch  3/10  train_loss 1.2774  train_acc 0.5374  val_loss 1.1653  val_acc 0.5822  (38.0s)
[resnet18 seed0] epoch  4/10  train_loss 1.0219  train_acc 0.6338  val_loss 0.9615  val_acc 0.6538  (39.2s)
[resnet18 seed0] epoch  5/10  train_loss 0.8187  train_acc 0.7118  val_loss 0.7622  val_acc 0.7348  (40.1s)
[resnet18 seed0] epoch  6/10  train_loss 0.6780  train_acc 0.7615  val_loss 0.6805  val_acc 0.7656  (39.6s)
[resnet18 seed0] epoch  7/10  train_loss 0.5713  train_acc 0.7996  val_loss 0.5752  val_acc 0.8030  (39.4s)
[resnet18 seed0] epoch  8/10  train_loss 0.4802  train_acc 0.8358  val_loss 0.4994  val_acc 0.8304  (39.1s)
[resnet18 seed0] epoch  9/10  train_los

: 

## Results table

With a single seed this is just the raw numbers per config. Once `SEEDS` has 3 values,
group by config and take mean and standard deviation instead: the standard deviation is
what tells you whether a gap between two configs is a real effect or just seed noise.


In [1]:
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("history", "test_preds", "test_labels")} for r in results])
print(df[["config", "seed", "params", "test_accuracy", "test_macro_f1"]])

# With 3 seeds you would instead do:
# summary = df.groupby("config")[["test_accuracy", "test_macro_f1"]].agg(["mean", "std"])
# summary


NameError: name 'pd' is not defined

## Training curves and final comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for r in results:
    axes[0].plot(r["history"]["val_loss"], marker="o", label=r["config"])
    axes[1].plot(r["history"]["val_acc"], marker="o", label=r["config"])
axes[0].set_title("Validation loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.bar([r["config"] for r in results], [r["test_accuracy"] for r in results], color=["#4C72B0", "#DD8452"])
plt.ylabel("test accuracy")
plt.title("Simple CNN vs ResNet-18 (no augmentation, SGD+momentum)")
plt.ylim(0, 1)
plt.show()


## Confusion matrix and per-class precision/recall

The mid-term asks for these for at least your best and worst configuration.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5))
if len(results) == 1:
    axes = [axes]
for ax, r in zip(axes, results):
    cm = confusion_matrix(r["test_labels"], r["test_preds"])
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(r["config"])
plt.tight_layout()
plt.show()

for r in results:
    print(f"\n--- {r['config']} ---")
    print(classification_report(r["test_labels"], r["test_preds"], target_names=CLASSES))


## How this becomes the full mid-term design

Nothing about the pipeline above changes. What changes is `CONFIGS` and `SEEDS`, plus how
you read the results afterward.

**1. Architecture study needs a third level.** Right now `resnet18` is effectively your
baseline. Add one genuinely different family (not just a bigger CNN), for example
DenseNet-121 or a small ViT from `torchvision.models`, as a new `build_model` branch.

**2. Augmentation study is entirely missing.** Add two new transforms in
`build_train_transform` (for example `"crop_flip"` and `"crop_flip_randaugment"`), and two
new configs that use the baseline architecture and baseline optimizer but swap the
augmentation.

**3. Optimizer study is entirely missing.** Add two new branches in `build_optimizer`
(for example `"adam"`, `"adamw"`), and two new configs that use the baseline architecture
and baseline augmentation but swap the optimizer. Remember optimizers do not share a
sensible learning rate; either justify a shared rate or run a small per-optimizer grid on
the validation set, and say which you did.

**4. One-factor-at-a-time (OFAT), 7 configs total.** The baseline is trained once and
reused as one level in all three studies. Concretely, `CONFIGS` grows to:

```python
CONFIGS = {
    "baseline":         {"architecture": "resnet18",   "augmentation": "none",       "optimizer": "sgd_momentum"},
    "arch_simple_cnn":  {"architecture": "simple_cnn",  "augmentation": "none",       "optimizer": "sgd_momentum"},
    "arch_densenet":    {"architecture": "densenet121", "augmentation": "none",       "optimizer": "sgd_momentum"},
    "aug_crop_flip":    {"architecture": "resnet18",    "augmentation": "crop_flip",  "optimizer": "sgd_momentum"},
    "aug_randaugment":  {"architecture": "resnet18",    "augmentation": "crop_flip_randaugment", "optimizer": "sgd_momentum"},
    "opt_adam":         {"architecture": "resnet18",    "augmentation": "none",       "optimizer": "adam"},
    "opt_adamw":         {"architecture": "resnet18",   "augmentation": "none",       "optimizer": "adamw"},
}
```

Every row differs from `baseline` in exactly one column. That is the self-check the
assignment PDF names explicitly: if a row differs in two columns, it no longer isolates a
single factor.

**5. Seeds become `[0, 1, 2]`, not `[0]`.** 7 configs times 3 seeds is the required
21 training runs. Every accuracy and F1 number in the paper is then reported as
mean plus/minus standard deviation over the 3 seeds, computed per run then averaged
(never pooled predictions across seeds).

**6. Save a results file.** Append every run's config name, seed, parameter count, and
metrics to a list, then `pd.DataFrame(...).to_csv(...)` at the end, this is the
results CSV/JSON the mid-term wants alongside the paper and notebooks.

**7. What stays exactly as it is here.** The stratified train/val split (created once,
reused everywhere), the shared `train_one_config`/`evaluate` functions, early stopping on
validation loss, and never touching the test set until the very end. That discipline is
the "correct experimental setup" the marking scheme is checking for, and it is already
built into this notebook.
